# 04e — Bogotá SO2 network series

Per user request. SO2 is a primary combustion/industrial-emissions tracer, and a
secondary-aerosol precursor (SO2 -> sulfate, contributing to PM2.5 mass) -- no
definitional-subset relationship with either PM2.5 or O3, so no causal exclusion needed.

Source: `Data/COLOMBIA/Hourmean/SO2_hourmean.csv`, which already has an hourly
**city-network-mean** `Bogota` column (same pre-aggregated-by-someone-upstream format
used for Medellín's pollutant series in notebook 01) -- so QC here is: range check +
Chauvenet's criterion on the hourly network-mean series itself, then daily aggregation,
same as Medellín's approach (no per-station selection possible/needed, there's only one
series). Starts 2016 (checked directly from the source file).


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd

from common.qc import chauvenet_outlier_mask

RAW_PATH = PROJECT_ROOT / "Data" / "COLOMBIA" / "Hourmean" / "SO2_hourmean.csv"
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

raw = pd.read_csv(RAW_PATH)
raw["Dates"] = pd.to_datetime(raw["Dates"])
s = pd.to_numeric(raw.set_index("Dates")["Bogota"], errors="coerce")
print(f"so2: {s.notna().sum()} non-null hourly obs, {s.index.min()} to {s.index.max()}")


so2: 41754 non-null hourly obs, 2016-01-01 00:00:00 to 2022-12-31 00:00:00


## QC: physical range check + Chauvenet's criterion

In [2]:
SO2_RANGE = (0.0, 500.0)
bad = (s < SO2_RANGE[0]) | (s > SO2_RANGE[1])
if bad.any():
    print(f"[range check] so2: {int(bad.sum())} values outside {SO2_RANGE} -> NaN")
    s[bad] = np.nan

mask = chauvenet_outlier_mask(s)
if mask.any():
    print(f"[Chauvenet] so2: flagged {int(mask.sum())} hourly outliers")
    s[mask] = np.nan
print(f"so2: {s.notna().sum()} non-null hourly obs after QC")


[Chauvenet] so2: flagged 325 hourly outliers
so2: 41429 non-null hourly obs after QC


## Aggregate to daily mean (>=18/24 valid hours)

In [3]:
n_valid = s.notna().resample("D").sum()
daily_mean = s.resample("D").mean()
so2_daily = daily_mean.where(n_valid >= 18)

network = so2_daily.reset_index()
network.columns = ["date", "so2"]
print(network.shape)
print("non-null network-day count:", network["so2"].notna().sum())

network["year"] = network["date"].dt.year
comp = network.groupby("year")["so2"].apply(lambda x: x.notna().mean())
print("completeness by year (reported, not filtered -- single series):")
print(comp.round(2).to_dict())
network = network.drop(columns=["year"])


(2557, 2)
non-null network-day count: 1723
completeness by year (reported, not filtered -- single series):
{2016: 0.0, 2017: 0.0, 2018: 0.98, 2019: 0.8, 2020: 1.0, 2021: 1.0, 2022: 0.94}


## Save

In [4]:
out_path = OUT_DIR / "bogota_so2_daily.parquet"
network.to_parquet(out_path, index=False)
print("Saved:", out_path, network.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_so2_daily.parquet (2557, 2)
